# JARVISEO 검출기 파인튜닝 — 회차 학습 (#167)

**쓰는 법**: 런타임 → 런타임 유형 변경 → **T4 GPU** · **고용량 RAM** → 저장 → 런타임 → **모두 실행**.
중간에 Google Drive 연결 허락 창이 한 번 뜬다. 그 밖에 고를 것은 없다.

- 입력: 내 드라이브 `JARVISEO/JARVISEO_R3_data.zip` — 맥에서 `tools/auto_label.py import` · `build` 로 만든 학습 묶음
  - `yolo_r3`: R2 검수 라벨(학습 331장) + R3 검수 라벨(새 영상 d07~d09 198장). 검증 d05 는 R2 와 같은 사람 라벨이다
  - `prep_report.txt`: 묶음을 만들 때 찍힌 기록(검수 장수 · 종류별 박스 · 검수 시간)
- 결과(Drive `JARVISEO/`): `detector/runs/r3`(가중치 · 그래프) · **`R3_report.txt`(이 실행의 기록 전부)**
- 대조군은 따로 학습하지 않는다. 이미 학습한 R2(`detector/runs/r2`)가 "새 사진이 없는 R3"라 그대로 대조군이다
- 회차 결과는 덮어쓰지 않는다. 이미 있으면 셀 2 에서 멈춘다.
- 레포 · CLIP 은 필요 없다(ultralytics 만 설치한다). 다음 회차는 셀 1 의 `ROUND` · `RUNS` 만 바꾼다.

In [ ]:
# 1. 설정 · 도움 함수 · GPU 확인
import csv
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from pathlib import Path

ROUND = "r3"
# R2 의 학습 사진 · 라벨은 그대로 두고 새 사진 198장만 더해 한 번 학습한다.
# 대조군은 이미 학습한 R2 다.
# (R2 는 같은 사진에 학습 라벨만 다르게 두 번 돌렸다: r2 · r2auto)
RUNS = ["r3"]
DRIVE = Path("/content/drive/MyDrive/JARVISEO")
ZIP = DRIVE / f"JARVISEO_{ROUND.upper()}_data.zip"
WORK = Path("/content/work")  # 압축은 Colab 로컬 디스크에 푼다(Drive 보다 읽기가 빠르다)
REPORT = DRIVE / f"{ROUND.upper()}_report.txt"
ULTRALYTICS = "8.4.155"  # 맥에서 R0 · R1 을 잰 버전과 같게 고정한다
EPOCHS = 100


def note(text=""):
    """화면에 찍고 보고서 파일에도 남긴다."""
    print(text)
    with REPORT.open("a", encoding="utf-8") as f:
        f.write(text + "\n")


if shutil.which("nvidia-smi") is None:
    raise RuntimeError("GPU 가 없습니다. 런타임 유형을 T4 GPU 로 바꾸고 다시 모두 실행하세요.")
GPU = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()
print("GPU:", GPU)

In [ ]:
# 2. Drive 연결 · 덮어쓰기 막기 (허락 창이 뜨면 내 계정으로 허용)
from google.colab import drive

drive.mount("/content/drive")
if not ZIP.exists():
    raise RuntimeError(
        f"학습 묶음이 없습니다: {ZIP} — 맥에서 만든 zip 을 내 드라이브 JARVISEO 에 올립니다."
    )
# 가중치가 이미 있으면 멈춘다. 덮어쓰면 그 회차를 같은 조건으로 다시 비교할 수 없다.
# 앞 실행이 중간에 끊겨 폴더만 남았으면, Drive 에서 그 폴더 이름을 바꿔(r2_failed1 등) 두고
# 다시 돌린다.
done = [DRIVE / "detector" / "runs" / run for run in RUNS]
if any(p.exists() for p in done):
    raise RuntimeError(
        "이미 결과가 있습니다(덮어쓰지 않음): " + ", ".join(str(p) for p in done if p.exists())
    )
if REPORT.exists():  # 앞 실행이 결과를 남기기 전에 멈췄다 → 그 기록은 이름을 바꿔 둔다
    REPORT.rename(REPORT.with_name(f"{REPORT.stem}_failed_{int(time.time())}.txt"))

In [ ]:
# 3. 설치 · 압축 풀기 (1~2분)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}"], check=True
)
if not WORK.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(WORK)
for run in RUNS:
    if not (WORK / f"yolo_{run}" / "data.yaml").exists():
        raise RuntimeError(f"zip 안에 yolo_{run}/data.yaml 이 없습니다")

note(f"# JARVISEO 검출기 {ROUND.upper()} 기록")
note(f"- 시각(UTC): {datetime.now(UTC):%Y-%m-%d %H:%M}")
note(f"- ultralytics {ULTRALYTICS} · GPU {GPU} · CPU {os.cpu_count()}개(데이터 준비 워커 기본 8)")
note(f"- 학습: yolov8n.pt 에서 · imgsz 640 · {EPOCHS} epoch · patience 0 · seed 0 · deterministic")
note()
note("## 학습 묶음 (맥에서 만든 기록)")
note((WORK / "prep_report.txt").read_text(encoding="utf-8"))

In [ ]:
# 4. 학습 두 번 (각 5분 안팎). 결과는 Drive detector/runs/<이름> 에 바로 저장된다(끊겨도 남음)
from ultralytics import YOLO

train_minutes = {}
for run in RUNS:
    started = time.time()
    # 지금 비서가 쓰는 가중치와 같은 구조에서 시작한다 → 회차 차이가 데이터 몫이 된다
    YOLO("yolov8n.pt").train(
        data=str(WORK / f"yolo_{run}" / "data.yaml"),
        epochs=EPOCHS,
        # 0 = 조기 종료 없음(ultralytics/utils/torch_utils.py:1112). 학습이 몇 분이라 아낄 게 없고,
        # 끝까지 돌아야 학습률이 다 내려가고 마지막 10 epoch 의 모자이크 끄기(close_mosaic)도 탄다.
        # best.pt 는 그대로 검증 점수로 고른다.
        patience=0,
        imgsz=640,
        seed=0,
        deterministic=True,
        project=str(DRIVE / "detector" / "runs"),
        name=run,
    )
    train_minutes[run] = (time.time() - started) / 60

In [ ]:
# 5. 두 best.pt 를 같은 검증 라벨(사람이 검수한 d05)로 재고 보고서 마무리
val_data = (
    WORK / f"yolo_{RUNS[0]}" / "data.yaml"
)  # 검증 사진 · 라벨은 두 묶음이 같다(build --val-labels)
for run in RUNS:
    run_dir = DRIVE / "detector" / "runs" / run
    with (run_dir / "results.csv").open(encoding="utf-8") as f:
        rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    # best.pt 를 고르는 기준(fitness)이 mAP50-95 다(ultralytics/utils/metrics.py:1011)
    best = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
    metrics = YOLO(str(run_dir / "weights" / "best.pt")).val(
        data=str(val_data), imgsz=640, plots=False, project=str(WORK / "val"), name=run
    )
    box = metrics.box
    note()
    note(f"## {run} (best.pt · 검증 d05 사람 라벨)")
    note(
        f"- 돈 epoch {len(rows)} / {EPOCHS} · best epoch {int(float(best['epoch']))} · "
        f"걸린 시간 {train_minutes[run]:.1f}분"
    )
    note(
        f"- 전체: P {box.mp:.3f} · R {box.mr:.3f} · mAP50 {box.map50:.3f} · mAP50-95 {box.map:.3f}"
    )
    note("| 종류 | 사진 | 박스 | P | R | mAP50 | mAP50-95 |")
    note("|---|---|---|---|---|---|---|")
    for row in metrics.summary():
        note(
            f"| {row['Class']} | {row['Images']} | {row['Instances']} | {row['Box-P']:.3f} | "
            f"{row['Box-R']:.3f} | {row['mAP50']:.3f} | {row['mAP50-95']:.3f} |"
        )
    note(f"가중치: {run_dir / 'weights' / 'best.pt'}")
note()
note("회차 비교(후보 재현율 · 정답률)는 맥에서 run_ablation.py --weights 로 R0 · R1 과 같이 잰다.")
print("\n끝. Drive JARVISEO/" + REPORT.name + " 을 내려받아 Claude 에게 보낸다.")

## 끝나면

- 맨 아래에 `끝.` 이 찍혔으면 성공이다.
  1. Drive `JARVISEO/R3_report.txt` 를 내려받아 Claude 에게 보낸다.
  2. Drive `JARVISEO/detector/runs/r3/weights/best.pt` → 레포 `data/models/detector_r3.pt` 로 내려받는다(파일 이름이 `best.pt` 라 받자마자 이름을 바꾼다).
- 중간에 빨간 오류로 멈췄으면 고치지 말고, 오류가 난 셀 번호와 마지막 20줄을 그대로 보낸다.